# Arquitectura de la Aplicación: Convivio

Convivio es una app para administración residencial, dividida en dos componentes principales: la aplicación móvil (**Frontend** en React Native / Expo) y el servidor central (**Backend** en Flask + PostgreSQL).

---

## 1. Vista General del Sistema

```
┌──────────────────────────────────────────────────┐
│               FRONTEND (Expo / React Native)      │
│                                                  │
│  Pantallas → Contextos → Servicios → api.js      │
│                    ↕ cache local en memoria       │
└────────────────────────┬─────────────────────────┘
                         │ HTTP (fetch JSON)
                         │ puerto 5000
┌────────────────────────▼─────────────────────────┐
│               BACKEND (Flask)                     │
│                                                  │
│  Controllers → Services → Repositories → DB       │
└────────────────────────┬─────────────────────────┘
                         │ SQLAlchemy ORM
┌────────────────────────▼─────────────────────────┐
│        PostgreSQL (Docker, puerto 5432)           │
└──────────────────────────────────────────────────┘
```

---

## 2. Roles de Usuario

La app soporta tres roles con navegación y funcionalidades distintas:

| Rol | Pantalla principal | Capacidades |
|---|---|---|
| **Residente** | `HomeScreen` | Reservas de áreas, PQR (peticiones/quejas/reclamos), chat con recepción, anuncios |
| **Administrador** | `AdminHomeScreen` | Gestión de múltiples conjuntos, aprobación de miembros, áreas comunes, anuncios, responder PQR |
| **Guarda** | `GuardReceptionScreen` | Chat con residentes (bandeja tipo Messenger), control de acceso |

El rol se almacena en `user.role` dentro de `AuthContext` y determina qué stack de navegación carga la app.

---

## 3. Sistema de Conjuntos Residenciales

Un **conjunto** es la unidad central de la app (el edificio o conjunto residencial).

### Registro de usuarios
1. El administrador **crea** un conjunto → el backend genera un código de 6 caracteres (ej. `CONV01`).
2. Residentes y guardas **ingresan ese código** para solicitar unirse (`JoinRequest`).
3. El administrador **aprueba o rechaza** las solicitudes desde `AdminMembersScreen`.
4. Un administrador puede gestionar **múltiples conjuntos** (los ve como tarjetas en `AdminHomeScreen`).

### Estado del conjunto en sesión
El usuario activo lleva en `AuthContext.user`:
- `conjuntoId`: ID del conjunto actualmente seleccionado
- `conjuntoIds[]`: todos los conjuntos que administra
- `conjuntoCode`: código para compartir
- `conjuntoName`: nombre para mostrar

---

## 4. Frontend: Capas y Responsabilidades

### 4.1 `api.js` — Comunicación HTTP base

**Archivo:** `front/src/services/api.js`

Centraliza todas las llamadas HTTP. Detecta automáticamente la IP del servidor:
- En **dispositivo físico** (Expo Go): usa la IP del bundler Metro (`Constants.expoConfig.hostUri`)
- En **emulador**: usa `localhost`

Expone `api.get()`, `api.post()`, `api.put()`, `api.delete()`. Todos los servicios lo usan exclusivamente.

### 4.2 Servicios — Cache local con pub/sub

Cada servicio sigue el mismo patrón de módulo con **cache en memoria**, **carga perezosa** y **suscripciones reactivas**:

```
let datos = [];          // cache local
let _loaded = false;     // flag de carga única
const subs = new Set();  // suscriptores para re-renders

export async function load() {
  if (_loaded) return;   // solo carga una vez por sesión
  _loaded = true;
  datos = await api.get('/endpoint');
  notify();              // avisa a todos los contextos
}

export function subscribe(fn) { subs.add(fn); return () => subs.delete(fn); }
```

Las **escrituras son optimistas**: primero actualizan el cache local, luego sincronizan con el backend. Esto hace la UI inmediata sin esperar la red.

| Servicio | Archivo | Datos que maneja | Cargado por |
|---|---|---|---|
| `conjuntoService` | `services/conjuntoService.js` | Lista de conjuntos, solicitudes de unión | `AuthContext` (al iniciar y después de login) |
| `bookingService` | `services/bookingService.js` | Reservas de áreas comunes, anuncios | `BookingContext` |
| `pqrService` | `services/pqrService.js` | Tickets PQR (peticiones/quejas/reclamos) | `PqrContext` |
| `areasService` | `services/areasService.js` | Configuración de áreas (habilitada/deshabilitada, foto, requiere aprobación) | Por pantalla, bajo demanda |
| `notificationsService` | `services/notificationsService.js` | Estado de lectura de notificaciones | `NotificationsContext` |

### 4.3 Contextos — Estado global de React

Los contextos consumen los servicios y exponen datos a las pantallas. Se anidan en orden:

```
AuthProvider
  └─ BookingProvider
       └─ PqrProvider
            └─ NotificationsProvider
                 └─ ReceptionChatProvider
                      └─ AccessibilityProvider
```

| Contexto | Responsabilidad |
|---|---|
| `AuthContext` | Usuario logueado, login/register/logout, selección de conjunto activo. Al montar: llama `conjuntoService.load()`. Tras login: llama `conjuntoService.reload()`. |
| `BookingContext` | Expone reservas y calendario. Llama `bookingService.load()` al montar. |
| `PqrContext` | Expone tickets PQR. Llama `pqrService.load()` al montar. |
| `NotificationsContext` | Agrega notificaciones de reservas y PQR. Se recalcula cuando cambia `bookingTick` o `pqrTick`. |
| `ReceptionChatContext` | Chat entre residente y guarda. **Completamente en memoria** (sin backend), datos de demo. |
| `AccessibilityContext` | Ajustes de accesibilidad (tamaño de fuente, reducción de movimiento). |

---

## 5. Flujo completo: Crear una Reserva

Este es el ciclo de vida cuando un residente reserva un área común:

**Paso A — Pantalla (`AreaFormScreen`):**
El usuario selecciona área, fecha y horario, y toca "Confirmar Reserva".
La pantalla llama a `bookingService.submitReservation({ areaName, year, day, timeSlot, ... })`.

**Paso B — Servicio (`bookingService.js`):**
1. Llama a `api.post('/bookings/', { ... })`.
2. **Actualiza el cache local** inmediatamente con la nueva reserva.
3. Si el área no requiere aprobación, también agrega un anuncio local.
4. Llama a `notify()` → todos los contextos suscritos se actualizan.

**Paso C — Backend: Controller (`booking_controller.py`):**
Recibe el JSON en `POST /api/bookings/`, extrae y valida los campos, delega al servicio.

**Paso D — Backend: Service (`booking_service.py`):**
Aplica lógica de negocio: ¿el área requiere aprobación del admin? → `status = 'pending_approval'` o `'confirmed'`.
Construye el objeto `Reservation` y lo pasa al repositorio.

**Paso E — Backend: Repository + PostgreSQL:**
`BookingRepository.save()` hace `db.session.commit()`. La reserva queda persistida.

**Paso F — Respuesta:**
El controlador devuelve JSON `201 Created`. El servicio frontend recibe la confirmación, actualiza el cache con el objeto del servidor y vuelve a llamar `notify()`.

---

## 6. Backend: Capas y Endpoints

### Estructura de capas
```
backend/src/
├── app.py                  # Fábrica Flask (crea app, registra blueprints, CORS)
├── config/
│   ├── config.py           # Variables de entorno (DATABASE_URL, SECRET_KEY)
│   └── extensions.py       # Instancias de SQLAlchemy y Flask-Migrate
├── models/__init__.py      # Modelos ORM (tablas de PostgreSQL)
├── controllers/            # Rutas HTTP (Blueprints de Flask)
├── services/               # Lógica de negocio
└── repositories/           # Acceso a base de datos
```

### Tablas en PostgreSQL

| Tabla | Modelo | Descripción |
|---|---|---|
| `users` | `User` | Usuarios con rol, apt, hash de contraseña, `conjunto_id` |
| `conjuntos` | `Conjunto` | Conjuntos residenciales con código único, dirección, foto |
| `join_requests` | `JoinRequest` | Solicitudes de unión a un conjunto (pending/approved/rejected) |
| `reservations` | `Reservation` | Reservas de áreas comunes por fecha y horario |
| `area_settings` | `AreaSetting` | Config por área: deshabilitada, requiere aprobación, foto |
| `tickets` | `Ticket` | PQR: peticiones, quejas y reclamos con respuesta del admin |
| `announcements` | `Announcement` | Anuncios creados por el administrador |

### Endpoints de la API (`/api/`)

| Prefijo | Controller | Operaciones principales |
|---|---|---|
| `/auth/` | `auth_controller.py` | `POST /login` |
| `/conjuntos/` | `conjunto_controller.py` | CRUD conjuntos, solicitudes de unión (requests), aprobación/rechazo |
| `/bookings/` | `booking_controller.py` | Reservas, anuncios, aprobar/cancelar reservas |
| `/pqr/` | `pqr_controller.py` | Tickets, responder ticket |
| `/areas/` | `area_controller.py` | Settings de áreas comunes (config + foto) |

### Datos de demo (run.py)

Al arrancar, `run.py` ejecuta automáticamente:
1. `db.create_all()` — crea todas las tablas si no existen.
2. `migrate_schema()` — agrega columnas nuevas con `ALTER TABLE ... IF NOT EXISTS`.
3. `seed_conjuntos()` — inserta conjuntos demo (`C001`: El Prado, `C002`: Torres del Parque).
4. `AuthService.create_demo_users()` — inserta usuarios demo:

| Usuario | Contraseña | Rol | Conjunto |
|---|---|---|---|
| `admin` | `1234` | administrador | C001 - El Prado |
| `prueba` | `1234` | residente | C001 - El Prado |
| `guarda` | `1234` | guarda | C001 - El Prado |

---

## 7. Funcionalidades Notables

### Calendario de Reservas
- `bookingService` calcula disponibilidad **localmente** desde el cache.
- Horarios fijos: 7 franjas por día (08-10, 10-12, 14-15, 15-16, 16-18, 18-20, 20-22).
- Un día completo se marca como "no disponible" cuando las 7 franjas están ocupadas.

### Chat de Recepción (Guarda ↔ Residentes)
- **Sin backend** — todo en `ReceptionChatContext` en memoria.
- El guarda tiene una **bandeja tipo Messenger** con múltiples conversaciones.
- Respuestas automáticas con delay de 12-15 segundos para simular respuestas reales.
- Al entrar a la app, se dispara un mensaje de bienvenida automático tras 8 segundos.

### Notificaciones
- `NotificationsContext` agrega notificaciones de **reservas** y **PQR** en una sola lista.
- Se recalcula automáticamente cuando `bookingTick` o `pqrTick` cambian (patrón reactivo).
- El estado de lectura/no-leído se guarda en `notificationsService` (en memoria).

### Fotos de Áreas y Conjuntos
- Las fotos se seleccionan de la galería del dispositivo con `expo-image-picker`.
- La URI local se guarda en el backend como texto (`photo_url` en conjuntos, `photo_uri` en áreas).
- La UI carga la imagen con `{ uri: photoUri }` en `expo-image`.

---

## 8. Cómo ejecutar el proyecto

Da doble clic en **`iniciar.bat`** en la raíz del proyecto. Este archivo automatiza:

1. **`docker-compose up -d`** — levanta PostgreSQL en el puerto 5432 (requiere Docker Desktop abierto).
2. **`python run.py`** — activa el entorno virtual, inicializa la BD con datos demo y arranca Flask en el puerto 5000.
3. **`npm start`** — en `front/`, levanta el bundler de Expo.

Escanea el QR con **Expo Go** en tu celular. La app detecta automáticamente la IP de tu computadora — no necesitas cambiar nada en el código.